In [1]:
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import HumanMessage
import os
import base64
import io
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\akhil\Documents\Multimodal_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv,find_dotenv
load_dotenv()  # Load environment variables from .env file
os.environ["OPENAI_API_KEY"] = os.getenv('OPEN_API_KEY')
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

c:\Users\akhil\Documents\Multimodal_RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
def embed_image(image_data):
    if isinstance(image_data,str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()

In [ ]:
#For creating the embeddings for text
def embed_text(text):
    inputs = clip_processor(text = text,
                            return_tensors="pt",
                            padding = True,
                            truncation = True,
                            max_length = 77)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        features = features/features.norm(dim = -1, keepdim=True)
    return features.squeeze().numpy()

In [8]:
import chunk
import fitz
pdf_path = "bar_graph_report.pdf"
doc = fitz.open(pdf_path)
all_docs = []
all_embeddings = []
image_data_store = {}
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)


C:\Users\akhil\AppData\Local\Temp\ipykernel_17172\3797798034.py:1: DeprecationWarning: 'chunk' is deprecated and slated for removal in Python 3.13
  import chunk


In [9]:
for i,page in enumerate(doc):
    ## process text
    text = page.get_text()
    if text.strip():
        ##temporary document for splitting
        temp_doc = Document(page_content=text, metadata = {"page":i, "type":"text"})
        text_chunks = splitter.split_documents([temp_doc])
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)
    
    ## process image
    ## Three important actions:
    '''1. convert pdf image to PIL format
    2. store as base64 for GPT-4V
    3. create CLIP embedding for retrieval'''
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_id = f"page_{i}_img_{img_index}"
            buffered = io.BytesIO()
            pil_image.save(buffered, format = "PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            image_doc = Document(
                page_content = f"[Image: {image_id}]",
                metadata = {"page": i, "type":"image", "image_id": image_id}
            )
            all_docs.append(image_doc)
        except Exception as e:
            print(f"Error processing image {image_id} on page {i}: {e}")
            continue
doc.close()

In [10]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Category A\nCategory B\nCategory C\nCategory D\nCategory E\nCategories\n0\n20\n40\n60\n80\nValues\n20\n35\n55\n70\n90\nProgressive Growth Bar Chart\nBar Graph Analysis Report\nThis bar graph displays a progressive upward trend across five distinct categories or time periods. \nEach successive bar demonstrates a clear increase in value compared to the previous one, indicating\nconsistent growth or improvement.\nThe visualization effectively illustrates the magnitude of change between each measurement point.'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='The visualization effectively illustrates the magnitude of change between each measurement point.\nSuch ascending patterns are commonly observed in business metrics like sales growth, user adoption, or\nperformance indicators.\nThe uniform spacing and proportional scaling make the data relationships easily interpretable for\ndecision-making purposes.')]

In [11]:
embeddings_array = np.array(all_embeddings)
vector_store = FAISS.from_embeddings(
    text_embeddings= [(doc.page_content,emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding = None,
    metadatas = [doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [12]:
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000027174B627B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000271751702C0>, root_client=<openai.OpenAI object at 0x0000027170CBEB40>, root_async_client=<openai.AsyncOpenAI object at 0x0000027174CE7DD0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [13]:
def retrieve_multimodal(query, k=3):
    query_embedding = embed_text(query)
    results = vector_store.similarity_search_by_vector(embedding=query_embedding, k=k)
    return results

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    content = []
    content.append({
        "type" : "text",
        "text": f"Question: {query}\n\nContext :\n"
    })
    
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    if text_docs:
        text_context = "\n\n".join([f"[Page {doc.metadata['page']}]: {doc.page_content}" for doc in text_docs])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url" : {
                    "url" : f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    content.append({
        "type": "text",
        "text": "\n\nPlease answer question based on provided images and text"
    })
    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    message = create_multimodal_message(query, context_docs)
    response = llm.invoke([message])
    print(f"\n{len(context_docs)} documents retrieved.")
    for doc in context_docs:
        doc_type = doc.metadata.get("type","unknown")
        page = doc.metadata.get("page","?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content)>100 else doc.page_content
            print(f" - Text from page {page}: {preview}")
        else:
            print(f" - Image from page {page}")
    print("\n")
    return response.content

In [19]:
if __name__ == "__main__":
    query = "summarize the data in the document"
    print(f"\nQuery: {query}")
    print("-"*50)
    print(f"Answer: {multimodal_pdf_rag_pipeline(query)}")
    print("="*70)


Query: summarize the data in the document
--------------------------------------------------
[{'type': 'text', 'text': 'Question: summarize the data in the document\n\nContext :\n'}, {'type': 'text', 'text': 'Text excerpts:\n[Page 0]: The visualization effectively illustrates the magnitude of change between each measurement point.\nSuch ascending patterns are commonly observed in business metrics like sales growth, user adoption, or\nperformance indicators.\nThe uniform spacing and proportional scaling make the data relationships easily interpretable for\ndecision-making purposes.\n\n[Page 0]: Category A\nCategory B\nCategory C\nCategory D\nCategory E\nCategories\n0\n20\n40\n60\n80\nValues\n20\n35\n55\n70\n90\nProgressive Growth Bar Chart\nBar Graph Analysis Report\nThis bar graph displays a progressive upward trend across five distinct categories or time periods. \nEach successive bar demonstrates a clear increase in value compared to the previous one, indicating\nconsistent growth o